# Out-of-sample evaluation of the statsmodels Logit

Evaluate the frozen, unweighted and unpenalized training fit behind `summary2()` against the frozen tuned L1/class-weighted logistic predictor. Both use the same 81 predictors, existing regenerated features, and original 290/103/80 chronological event split. Load the exported full-precision coefficients and training scaling; **no refitting, tuning, or model reselection**.

Use the same event-ranking metrics and Top@1 classification rule as the paper. A separate 10,000-draw paired event bootstrap compares these two frozen fits. This additional contrast is exploratory and is not part of the paper's full 66-comparison family. Neither training pseudo-R² nor AIC is held-out predictive performance.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import hashlib
import json
import sys

import numpy as np
import pandas as pd
from scipy.special import expit
from IPython.display import display
from pumps_and_dumps.core.paths import get_root_dir, FEATURE_DIR

PROJECT_ROOT = get_root_dir()
sys.path.insert(0, str(PROJECT_ROOT))
from backtest.pipelines.LogisticRegression.pipe import LogisticRegressionPipeline
from backtest.utils.sample import DatasetType
from backtest.utils.experiment import Experiment
from backtest.utils.metrics import calculate_topk_percent_auc, calculate_topk, calculate_topk_percent
from backtest.utils.evaluation import evaluate_experiments_classification
from pumps_and_dumps.backtest.robustness.auc_bootstrap import evaluate_test_auc_bootstrap

PAPER_RUN = PROJECT_ROOT / "notebooks/analysis_outputs/paper_0167ddd_473_l2/paper_run"
TABLES = PAPER_RUN / "tables"
OUTPUT = PROJECT_ROOT / "notebooks/analysis_outputs/paper_0167ddd_473_l2/statsmodels_evaluation"
OUTPUT.mkdir(parents=True, exist_ok=True)
feature_hashes = pd.read_csv(PAPER_RUN / "feature_hashes.csv")
current_hashes = feature_hashes["file"].map(lambda name: hashlib.sha256((FEATURE_DIR / "pumps" / name).read_bytes()).hexdigest())
if not current_hashes.equals(feature_hashes["sha256"]):
    raise ValueError("Features differ from those used for the saved fits")

sample = LogisticRegressionPipeline().create_sample()
scaling = pd.read_csv(TABLES / "logistic_statsmodels_scaling.csv", index_col=0, float_precision="round_trip")
sm_parameters = pd.read_csv(TABLES / "logistic_statsmodels_coefficients.csv", index_col=0, float_precision="round_trip")["Coef."]
predictive_parameters = pd.read_csv(TABLES / "logistic_coefficients.csv", index_col=0, float_precision="round_trip")["coefficient"]
features = scaling.index

def frozen_logistic_model(coefficient, intercept, mean, sd):
    def rank(dataset):
        X = dataset.get_data().loc[:, coefficient.index].astype(float)
        return expit(((X - mean) / sd) @ coefficient + intercept).to_numpy()
    return SimpleNamespace(rank=rank)

statsmodels_model = frozen_logistic_model(sm_parameters.loc[features], sm_parameters["const"], scaling.training_mean, scaling.training_sd)
predictive_model = frozen_logistic_model(predictive_parameters.loc[features], predictive_parameters["intercept"], 0, 1)
models = [
    Experiment("Statsmodels Logit (unweighted, unpenalized)", statsmodels_model, sample),
    Experiment("Logistic regression + Tuned", predictive_model, sample),
]


In [ ]:
# Verify reconstruction against the recorded training likelihood and test predictions.
training = sample.get_dataset(DatasetType.TRAIN)
X_train = (training.get_data().loc[:, features] - scaling.training_mean) / scaling.training_sd
linear_train = X_train @ sm_parameters.loc[features] + sm_parameters["const"]
y_train = training.get_label().astype(int)
reconstructed_log_likelihood = float((y_train * linear_train - np.logaddexp(0, linear_train)).sum())
recorded_diagnostics = json.loads((TABLES / "logistic_diagnostics.json").read_text())
if not np.isclose(reconstructed_log_likelihood, recorded_diagnostics["log_likelihood"], rtol=0, atol=1e-8):
    raise ValueError("Statsmodels reconstruction does not match its recorded training fit")

saved_predictions = pd.read_parquet(PROJECT_ROOT / "notebooks/analysis_outputs/paper_0167ddd_473_l2/test_auc_bootstrap/test_predictions.parquet")
saved_logistic = saved_predictions.loc[saved_predictions.model == "Logistic regression + Tuned"].sort_values("row")
reconstructed_predictions = predictive_model.rank(sample.get_dataset(DatasetType.TEST))
if not np.allclose(reconstructed_predictions, saved_logistic.probas_pred, rtol=0, atol=1e-12):
    raise ValueError("Predictive logistic reconstruction differs from the recorded test fit")
print(f"Verified frozen statsmodels likelihood: {reconstructed_log_likelihood:.8f}")
print("Verified tuned logistic test predictions and all 473 feature hashes.")


In [ ]:
def evaluate_split(kind):
    classification = evaluate_experiments_classification(models, dataset_type=kind)
    rows = []
    for experiment in models:
        dataset = experiment.get_sample().get_dataset(kind)
        model = experiment.get_model()
        topk = calculate_topk(model, dataset, bins=[1, 5, 10, 20, 30])
        topkp = calculate_topk_percent(model, dataset, bins=[0.01, 0.02, 0.05, 0.1, 0.2])
        rows.append({
            "split": kind.name, "model": experiment.get_experiment_name(),
            "events": dataset.all_data().pump_hash.nunique(), "rows": len(dataset.all_data()),
            "Top@k%AUC": calculate_topk_percent_auc(model, dataset),
            **{f"Top@{k}": value for k, value in topk.items()},
            **{f"Top@{100 * k:g}%": value for k, value in topkp.items()},
            **classification.loc[experiment.get_experiment_name()].to_dict(),
        })
    return pd.DataFrame(rows)

performance = pd.concat([evaluate_split(DatasetType.VALIDATION), evaluate_split(DatasetType.TEST)], ignore_index=True)
performance.to_csv(OUTPUT / "performance.csv", index=False)
display(performance.set_index(["split", "model"]))


In [ ]:
bootstrap = evaluate_test_auc_bootstrap(models, dataset_type=DatasetType.TEST, n_bootstrap=10_000, random_state=42)
bootstrap.summary.to_csv(OUTPUT / "auc_summary.csv")
bootstrap.pairwise.to_csv(OUTPUT / "auc_pairwise.csv", index=False)
bootstrap.bootstrap_aucs.to_csv(OUTPUT / "auc_distributions.csv")
bootstrap.event_aucs.to_csv(OUTPUT / "event_aucs.csv")
np.savez_compressed(OUTPUT / "resampled_events.npz", sampled_indices=bootstrap.sampled_indices, event_ids=bootstrap.event_aucs.index.to_numpy(dtype=str))
pd.concat(bootstrap.scored_datasets, names=["model", "row"]).reset_index().to_parquet(OUTPUT / "test_predictions.parquet", index=False)
metadata = {
    "comparison": "Exploratory two-fit comparison; supplement to the full 78-pair comparison",
    "method": "10,000 paired whole-test-event draws, seed 42; fixed models, independent events assumed",
    "refitted": False, "feature_generation": False,
    "source_hashes": {str(path.relative_to(PROJECT_ROOT)): hashlib.sha256(path.read_bytes()).hexdigest() for path in [
        TABLES / "logistic_statsmodels_scaling.csv", TABLES / "logistic_statsmodels_coefficients.csv",
        TABLES / "logistic_coefficients.csv", PAPER_RUN / "feature_hashes.csv",
        PROJECT_ROOT / "notebooks/statsmodels_evaluation.ipynb",
    ]},
}
(OUTPUT / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
display(bootstrap.summary)
display(bootstrap.pairwise)
